# Distil Toolforge's LLM judge into a cross-encoder reranker

Toolforge decides *reuse an existing tool vs. build a new one* with an LLM judge (97% correct on the
hand-labelled set, about 870 tokens and 2.5 s per decision). This notebook fine-tunes a small
cross-encoder on the judge's own decisions and measures how much of that accuracy it keeps at zero tokens.

**Before running:** build the dataset on your machine (`python -m evals.distill_data`) and either push the
repo to GitHub (cell 2, option A) or upload the `toolforge_colab.zip` it writes (option B).

`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: training will run on CPU (slower)"

## 1. Get the code and dataset
Option A clones your public repo. Option B asks you to upload `toolforge_colab.zip`, which
`python -m evals.distill_data` writes to the repo root when it finishes.

In [ ]:
import os
import pathlib
import subprocess
import zipfile

REPO = "https://github.com/chandrasekharthapa/toolforge.git"
os.chdir("/content")
ok = False
if not pathlib.Path("toolforge/evals/distill/pairs_train.jsonl").exists():
    ok = subprocess.run(["git", "clone", "-q", REPO]).returncode == 0 and pathlib.Path(
        "toolforge/evals/distill/pairs_train.jsonl").exists()
    if not ok:
        print("Clone failed or the dataset is not in the repo; upload toolforge_colab.zip instead.")
        from google.colab import files
        up = files.upload()
        name = next(iter(up))
        zipfile.ZipFile(name).extractall("toolforge")
os.chdir("/content/toolforge")
print(sorted(p.name for p in pathlib.Path("evals/distill").iterdir()))

In [ ]:
!pip install -q "transformers>=4.40"

## 2. Train and evaluate
Zero-shot baseline first, then fine-tuning; the threshold is chosen on validation tools never seen in
training, and the test set is the 65 hand-labelled needs over 32 tools that never appear in training.

In [ ]:
!python evals/train_reranker.py --epochs 4 --lr 3e-5 --batch 32

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open("evals/results/reranker.md").read()))

## 3. Download the model and results
Unzip `toolforge_reranker.zip` into your repo so that `evals/distill/reranker/` exists, and copy
`reranker.json` / `reranker.md` into `evals/results/`.

In [ ]:
!cd evals/distill && zip -qr /content/toolforge_reranker.zip reranker
!cp evals/results/reranker.json evals/results/reranker.md /content/
from google.colab import files

for f in ["/content/toolforge_reranker.zip", "/content/reranker.json", "/content/reranker.md"]:
    files.download(f)